In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer ,pipeline 
import torch
from pprint import pprint
import textwrap
from tqdm import tqdm

2025-11-06 22:59:24.371585: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-06 22:59:24.406543: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-06 22:59:25.396383: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
final_model = "./merged-model"

model = AutoModelForCausalLM.from_pretrained(final_model)
tokenizer = AutoTokenizer.from_pretrained(final_model)


# Now we can evaluate it , but how ?

We will the notion of **perplexity**.
# Perplexity

**Perplexity** is a measurement used in **Natural Language Processing (NLP)** and **probabilistic modeling** to evaluate the quality of a language model. It indicates how "uncertain" the model is when predicting the next unit (word, character, or token) in a sequence.

## Definition

For a language model predicting a sequence of words $ w_1, w_2, ..., w_N $, perplexity is defined as:

$$
\text{Perplexity} = PPL = 2^{-\frac{1}{N} \sum_{i=1}^{N} \log_2 P(w_i \mid w_1, ..., w_{i-1})}
$$

- $ P(w_i \mid w_1, ..., w_{i-1})$ : the probability the model assigns to word $w_i $ given the previous context.  
- $N$ : the total number of words in the sequence.

## Interpretation

- A **low perplexity** means the model predicts the next words well; it is less "perplexed."  
- A **high perplexity** means the model struggles to predict the sequence correctly.

## Intuition

Perplexity can be thought of as the **average branching factor**: how many choices the model considers equally likely at each step.  
- Example: A perplexity of 50 means that, on average, the model is as uncertain as if it had to choose between 50 equally likely options for the next word.

The lowest the perplexity is , the better is  the model !

# Test dataset 

Here I asked to chatgpt to generate 40 examples of differents queries to compute the perplixity of the model and evaluate it.

In [ ]:
test_ds = [
    "Patient is a 60-year-old woman with a history of hypertension. "
    "She presents with chest pain that started 2 hours ago and is radiating to her left arm. "
    "The pain is described as pressure-like (8/10). Given these risk factors and "
    "symptoms, the most concerning diagnosis is one of the following: "
    " { A) Acute myocardial infarction B) Angina pectoris C) Gastroesophageal reflux disease (GERD)}. Choose the best answer. "
    ,
    "Patient is a 70-year-old male with a history of diabetes and obesity. "
    "He presents with a headache that has been persistent for the last week, accompanied by blurred vision. "
    "His blood pressure is elevated at 160/90 mmHg. Given these risk factors and symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Hypertensive crisis B) Cluster headache C) Retinal detachment}."
    ,
    "Patient is a 45-year-old female with a history of frequent alcohol use. "
    "She presents with a 3-day history of nausea and vomiting, along with right upper quadrant pain. "
    "The pain worsens after meals. Given these risk factors and symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Acute pancreatitis B) Gallbladder disease (cholelithiasis) C) Peptic ulcer disease}."
    ,
    "Patient is a 55-year-old male who recently returned from a trip to Southeast Asia. "
    "He presents with a fever, chills, and muscle aches. His symptoms started 5 days ago. "
    "Given these risk factors and symptoms, the most concerning diagnosis is one of the following: "
    " { A) Dengue fever B) Malaria C) Influenza}."
    ,
    "Patient is a 30-year-old woman who presents with a 2-week history of a sore throat, fever, and fatigue. "
    "She also has swollen lymph nodes in her neck. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Mononucleosis B) Strep throat C) Tonsillitis}."
    ,
    "Patient is a 65-year-old male with a history of smoking and chronic cough. "
    "He presents with shortness of breath and a productive cough with yellow sputum. "
    "These symptoms have worsened over the past 2 weeks. Given these risk factors and symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Chronic obstructive pulmonary disease (COPD) exacerbation B) Pneumonia C) Lung cancer}."
    ,
    "Patient is a 40-year-old woman who presents with a new, sharp, left-sided abdominal pain that started suddenly 4 hours ago. "
    "She also has nausea and vomiting. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Ectopic pregnancy B) Appendicitis C) Ovarian torsion}."
    ,
    "Patient is a 75-year-old male with a history of atrial fibrillation. "
    "He presents with sudden onset of left-sided weakness and difficulty speaking. "
    "His blood pressure is 180/100 mmHg. Given these risk factors and symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Ischemic stroke B) Transient ischemic attack (TIA) C) Bell's palsy}."
    ,
    "Patient is a 50-year-old woman with a history of thyroid disease. "
    "She presents with weight gain, fatigue, and cold intolerance. "
    "These symptoms have gradually developed over the last 6 months. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Hypothyroidism B) Depression C) Chronic fatigue syndrome}."
    ,
    "Patient is a 25-year-old male with a history of anxiety. "
    "He presents with a racing heart, chest tightness, and dizziness that started 30 minutes ago. "
    "Given these symptoms, the most concerning diagnosis is one of the following: "
    " { A) Panic attack B) Acute myocardial infarction C) Anxiety disorder}."
    ,
    "Patient is a 60-year-old woman who presents with a 4-day history of low back pain that radiates down her left leg. "
    "She also has numbness in her left foot. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Sciatica B) Spinal stenosis C) Herniated disc}."
    ,
    "Patient is a 45-year-old male with a history of hypertension and obesity. "
    "He presents with swelling in both lower legs and shortness of breath. "
    "These symptoms have worsened over the past 2 days. Given these risk factors and symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Congestive heart failure B) Deep vein thrombosis C) Pulmonary embolism}."
    ,
    "Patient is a 50-year-old woman with a history of asthma. "
    "She presents with wheezing and increased difficulty breathing, which began 2 hours ago. "
    "She is using her inhaler more frequently. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Asthma exacerbation B) Pneumothorax C) Acute bronchitis}."
    ,
    "Patient is a 65-year-old male with a history of chronic kidney disease. "
    "He presents with swelling in his face and hands, along with decreased urine output. "
    "His blood pressure is 150/95 mmHg. Given these symptoms, the most concerning diagnosis is one of the following: "
    " { A) Acute renal failure B) Nephrotic syndrome C) Hypertensive nephropathy}."
    ,
    "Patient is a 35-year-old woman who presents with a 2-week history of headaches, dizziness, and neck stiffness. "
    "She has also been feeling nauseous and feverish. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Meningitis B) Tension headache C) Cervical spondylosis}."
    ,
    "Patient is a 40-year-old male who presents with a 1-week history of a cough and fever. "
    "He is also experiencing night sweats and unexplained weight loss. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Tuberculosis B) Pneumonia C) Sinusitis}."
    ,
    "Patient is a 28-year-old female with a history of chronic headaches. "
    "She presents with a severe headache (9/10) that started suddenly, along with nausea and vomiting. "
    "She has a history of oral contraceptive use. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Subarachnoid hemorrhage B) Migraine C) Cluster headache}."
    ,
    "Patient is a 60-year-old male smoker who presents with a persistent cough and hemoptysis. "
    "He also reports weight loss and fatigue. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Lung cancer B) Tuberculosis C) Chronic obstructive pulmonary disease (COPD)}."
    ,
    "Patient is a 55-year-old male with a history of diabetes. "
    "He presents with a painful, swollen left foot. He reports a history of poorly controlled blood sugar and numbness in his foot. "
    "Given these symptoms, the most concerning diagnosis is one of the following: "
    " { A) Diabetic foot ulcer B) Gout C) Cellulitis}."
    ,
    "Patient is a 70-year-old female with a history of osteoarthritis. "
    "She presents with joint pain and stiffness, particularly in the knees, which worsens in the morning. "
    "Given these symptoms, the most concerning diagnosis is one of the following: "
    " { A) Osteoarthritis B) Rheumatoid arthritis C) Gout}."
    ,
    "Patient is a 55-year-old male with a 2-week history of progressive fatigue and weight loss. "
    "He also has a low-grade fever and night sweats. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Leukemia B) Hodgkin lymphoma C) Tuberculosis}."
    ,
    "Patient is a 40-year-old woman with a history of multiple pregnancies. "
    "She presents with a sudden onset of severe right lower quadrant pain. "
    "She also reports nausea and vomiting. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Ovarian cyst rupture B) Appendicitis C) Ectopic pregnancy}."
    ,
    "Patient is a 60-year-old female with a history of chronic pain management. "
    "She presents with increased pain in her lower back and bilateral leg weakness. "
    "She has difficulty walking and reports numbness in her feet. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Spinal cord compression B) Osteoarthritis C) Peripheral neuropathy}."
    ,
    "Patient is a 45-year-old male with a history of heavy alcohol use. "
    "He presents with jaundice, abdominal pain, and dark urine. "
    "His liver enzymes are elevated. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Hepatitis B C) Alcoholic cirrhosis D) Gallstones}."
    ,
    "Patient is a 50-year-old female who presents with progressive shortness of breath and fatigue. "
    "She has a history of hypertension and is now noticing swelling in her ankles. "
    "Given these symptoms, the most concerning diagnosis is one of the following: "
    " { A) Heart failure B) Pulmonary embolism C) Chronic obstructive pulmonary disease (COPD)}."
    ,
    "Patient is a 28-year-old male who presents with painful urination, fever, and chills. "
    "He reports frequent urination and lower abdominal pain. "
    "Given these symptoms, the most concerning diagnosis is one of the following: "
    " { A) Urinary tract infection (UTI) B) Pyelonephritis C) Bladder cancer}."
    ,
    "Patient is a 33-year-old woman who presents with a rash on her face and joints. "
    "She reports a butterfly-shaped rash across her cheeks and nose. "
    "She also has joint pain and swelling in her hands. Given these symptoms, "
    "the most concerning diagnosis is one of the following: "
    " { A) Systemic lupus erythematosus B) Rheumatoid arthritis C) Psoriatic arthritis}."
]


# 1) Tokenize every sentance of the test_ds  

But before we need to determin the best length to padd the others sentances that have less tokens than this value.  

For this I decided to take the as value the **95-th percentile** of the list containing the length of every tokenized sentance.

In [ ]:
import numpy as np
tes = tokenizer(test_ds)["input_ids"]

tab_len = [len(sub) for sub in tes]
tab_num = np.array(tab_len)
mean = round(np.mean(tab_num))
percentile_95 = round(np.percentile(tab_len,95))
print(f'The mean number of tokens per sentance is : {mean} tokens')
print(f'The 95 th percentile of the lengths is : {percentile_95}')

MAX_TOKENS = int(percentile_95)

In [ ]:
def tokenize(ds):
    """ 
    This function tokenize the input dataset and returns a tensor by using as max_length: MAX_TOKENS 
    """
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer(ds,return_tensors="pt",max_length= MAX_TOKENS, truncation= True , padding= 'max_length')

tokenized_ds = tokenize(test_ds)

# 2 ) How to calculate the perplexity now ?

The formula is : 

$
\text{Perplexity} = PPL = e^{-\frac{1}{N} \sum_{i=1}^{N} \log_2 P(w_i \mid w_1, ..., w_{i-1})}
$

- $ P(w_i \mid w_1, ..., w_{i-1})$ : the probability the model assigns to word $w_i $ given the previous context.  
- $N$ : the total number of words in the sequence.

# But there is one problem !  

As you may know , LLMs have some limits, and one these limits is : **the window context**  

<u>**Quick explanation:**</u>  
A LLM uses the previous tokens that he generated to actually generate new tokens : that is the context . But this context is not unlimited , he can only **"attend to"** a limited number of token (this number depends on the model used).

In our case as you have seen we sum the $log_2 P(w_i \mid w_1, ..., w_{i-1})$ over the number of words . But the issue is that if we want to compute this for a long text that is longer than the size of the context window we will be forced to chunk the text and then at every new chunk , **the model will completly lose the previous context** and then the model will typically yield a higher (worse) PPL because the model will have less context at most of the prediction steps.

# So what is the solution ?  
The solution is to use a sliding window we will repeatedly slide the context window so that the model has more context when making each prediction.

(Here you can find a lovely animation https://huggingface.co/docs/transformers/perplexity)



In [ ]:
torch.cuda.empty_cache()
test_data = tokenized_ds.input_ids
device = "cuda" if torch.cuda.is_available() else "cpu"
stride = 70
w_context = 100  
seq_len = test_data.size(1)
nll_sum  = 0.0
n_tokens = 0
previous_end = 0

for start in tqdm(range(0,seq_len, stride)):
    end = min(seq_len,start+w_context)
    input = test_data[: , start:end].to(device)
    overlap = end - previous_end
    tgt = input.clone()
    #We mask the overlaped tokens to not compute their loss
    tgt[:,:-overlap] = -100
    
    with torch.no_grad():
        outputs= model(input, labels = tgt)
        neg_ll = outputs.loss
        
    num_valid_tokens = (tgt != -100).sum().item()
    batch_size = test_data.size(0)
    num_loss_tokens = num_valid_tokens - batch_size  # subtract batch_size due to internal label shift
    nll_sum += neg_ll * num_loss_tokens
    n_tokens += num_loss_tokens
    
    previous_end = end
    
    if end == seq_len:
        break
    
avg_nll = nll_sum/n_tokens
ppl = torch.exp(avg_nll)

    

In [ ]:
print(ppl)

# In reality we don't need to do this because our context window is sufficient (sorry) 

In [ ]:
def perplexity(ds):
    input_ids = ds.input_ids
    nll_sum = 0.0
    n_tokens = 0

    for t in input_ids:
        t = t.unsqueeze(0)
        t = t.to(device)
        labels = t.clone().unsqueeze(0)
        with torch.no_grad():
            outputs = model(t, labels=labels)
            neg_ll = outputs.loss
        
        num_tokens = (labels != -100).sum().item()
        nll_sum += neg_ll * num_tokens
        n_tokens += num_tokens

    avg_nll = nll_sum / n_tokens
    ppl = torch.exp(avg_nll)
    return ppl


result = perplexity(tokenized_ds)
print(f"The perplexity of our fine-tuned model is :{result.item()}")